# 단계 3 — YOLOv8n-cls Fine-Tuning (연기 탐지)

복강경 수술 영상에서 연기를 탐지하는 이진 분류 모델을 학습한다.

| 항목 | 내용 |
|---|---|
| 모델 | YOLOv8n-cls (ImageNet pretrained → fine-tuning) |
| 목표 | **Recall ≥ 0.99** (연기를 놓치는 것이 가장 치명적) |
| 데이터 | `data/smoke_cls/` — train 769장×2, val 192장×2 |
| 클래스 | `smoke` (gt 없음) / `no_smoke` (gt 붙음) |

## 0. 환경 확인

In [16]:
import torch
from pathlib import Path

# GPU 사용 가능 여부 확인
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 디바이스  : {device}")
if device == "cuda":
    print(f"GPU 이름       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# 데이터셋 경로 및 이미지 수 확인
BASE_DIR  = Path("..").resolve()
DATA_DIR  = BASE_DIR / "data" / "smoke_cls"

for split in ("train", "val"):
    for cls in ("smoke", "no_smoke"):
        count = len(list((DATA_DIR / split / cls).glob("*.png")))
        print(f"  {split:5s}/{cls:10s} : {count}장")

사용 디바이스  : cuda
GPU 이름       : NVIDIA GeForce RTX 4080 SUPER
VRAM           : 17.2 GB
  train/smoke      : 769장
  train/no_smoke   : 769장
  val  /smoke      : 192장
  val  /no_smoke   : 192장


## 1. Fine-Tuning 학습

**핵심 파라미터 설명:**
- `epochs=50` + `patience=10` → 10 epoch 동안 개선 없으면 조기 종료
- `imgsz=224` → YOLOv8n-cls 기본 입력 크기
- `batch=32` → GPU VRAM에 따라 조정 가능
- `device=0` → GPU 사용 (CPU면 `device='cpu'`로 변경)
- `workers=2` → **Windows에서 DataLoader 프로세스 수 제한 (기본 8 → CPU 과부하 방지)**

> 학습 완료 후 `runs/smoke_detector/yolov8n_cls/weights/best.pt` 에 최적 가중치 저장됨

In [17]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")

results = model.train(
    data      = str(DATA_DIR),
    epochs    = 50,
    imgsz     = 224,
    batch     = 32,
    device    = 0 if torch.cuda.is_available() else "cpu",
    patience  = 10,
    workers   = 0,
    cache     = "ram",
    project   = str(BASE_DIR / "runs" / "smoke_detector"),
    name      = "yolov8n_cls",
    exist_ok  = True,

    # ── 조명 불변성 확보를 위한 극단적 Augmentation ──────
    # 목표: 모델이 "절대 밝기" 대신 연기 고유의 텍스처/산란 패턴을 학습
    #
    # hsv_v=0.9: 밝기를 원본의 10%~190% 범위로 무작위 변경
    #   → 어두운 핀포인트 조명(mean≈90)부터 밝은 수술 영상(mean≈130)까지
    #     모두 학습 범위에 포함되어 밝기로는 smoke/no_smoke를 구분 못하게 함
    hsv_v     = 0.9,
    # hsv_s=0.7: 채도 변화로 연기의 탁한 색감 다양성 흡수
    hsv_s     = 0.7,
    # fliplr/flipud: 수술 방향/카메라 각도 무관하게 일반화
    fliplr    = 0.5,
    flipud    = 0.3,
    # translate/scale: 카메라 거리/위치 차이 흡수
    translate = 0.1,
    scale     = 0.3,
)

Ultralytics 8.4.37  Python-3.12.3 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=ram, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=F:\\Capstone_Project\AI_Capstone_Project\data\smoke_cls, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.9, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_cls, nbs=64, nms=False, opset=None, optimize=False, 

## 2. Val 세트 평가 및 Confusion Matrix

학습된 best.pt로 val 전체를 평가하고 클래스별 Recall을 확인한다.

In [18]:
import time
import random
from pathlib import Path

BEST_PT   = BASE_DIR / "runs" / "smoke_detector" / "yolov8n_cls" / "weights" / "best.pt"
CONF_THRES = 0.3   # pretrained_test.py와 동일한 threshold 유지

print(f"[로드] {BEST_PT}")
best_model = YOLO(str(BEST_PT))

# val 전체 이미지로 평가
val_smoke_imgs    = list((DATA_DIR / "val" / "smoke").glob("*.png"))
val_no_smoke_imgs = list((DATA_DIR / "val" / "no_smoke").glob("*.png"))

tp = fp = fn = tn = 0
total_ms = 0.0

# smoke 이미지 추론
for img in val_smoke_imgs:
    t0   = time.perf_counter()
    res  = best_model.predict(str(img), verbose=False)
    total_ms += (time.perf_counter() - t0) * 1000

    names     = res[0].names                          # {0: 'no_smoke', 1: 'smoke'}
    top1_idx  = int(res[0].probs.top1)
    pred_name = names[top1_idx]

    if pred_name == "smoke":
        tp += 1
    else:
        fn += 1   # ← 가장 위험한 오류 (연기 놓침)

# no_smoke 이미지 추론
for img in val_no_smoke_imgs:
    t0   = time.perf_counter()
    res  = best_model.predict(str(img), verbose=False)
    total_ms += (time.perf_counter() - t0) * 1000

    names     = res[0].names
    top1_idx  = int(res[0].probs.top1)
    pred_name = names[top1_idx]

    if pred_name == "no_smoke":
        tn += 1
    else:
        fp += 1

# 지표 계산
n_total   = tp + fp + fn + tn
accuracy  = (tp + tn) / n_total
precision = tp / max(tp + fp, 1)
recall    = tp / max(tp + fn, 1)
f1        = 2 * precision * recall / max(precision + recall, 1e-9)
avg_ms    = total_ms / n_total

print("\n" + "=" * 55)
print("  Fine-tuned YOLOv8n-cls 평가 결과 (val 전체)")
print("=" * 55)
print(f"  샘플 수        : smoke {len(val_smoke_imgs)}장, no_smoke {len(val_no_smoke_imgs)}장")
print(f"  conf threshold : {CONF_THRES}")
print(f"  TP={tp}  FP={fp}  FN={fn}  TN={tn}")
print(f"  Accuracy       : {accuracy:.4f}")
print(f"  Precision      : {precision:.4f}")
print(f"  Recall (smoke) : {recall:.4f}  ← 핵심 지표  (목표 ≥ 0.99)")
print(f"  F1 Score       : {f1:.4f}")
print(f"  평균 추론 속도 : {avg_ms:.1f} ms/프레임")
print("=" * 55)

[로드] F:\일\Capstone_Project\AI_Capstone_Project\runs\smoke_detector\yolov8n_cls\weights\best.pt

  Fine-tuned YOLOv8n-cls 평가 결과 (val 전체)
  샘플 수        : smoke 192장, no_smoke 192장
  conf threshold : 0.3
  TP=183  FP=2  FN=9  TN=190
  Accuracy       : 0.9714
  Precision      : 0.9892
  Recall (smoke) : 0.9531  ← 핵심 지표  (목표 ≥ 0.99)
  F1 Score       : 0.9708
  평균 추론 속도 : 7.6 ms/프레임


## 2-1. Precision-Recall Trade-off 분석

threshold를 0.05~0.95 범위로 바꾸며 Precision/Recall 변화를 측정한다.
원하는 운용 기준(예: Precision 우선 vs Recall 우선)에 맞는 threshold를 선택하자.

In [19]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── val 이미지별 smoke confidence 수집 ───────────────────
val_smoke_imgs    = list((DATA_DIR / "val" / "smoke").glob("*.png"))
val_no_smoke_imgs = list((DATA_DIR / "val" / "no_smoke").glob("*.png"))

smoke_idx = [k for k, v in best_model.names.items() if v == "smoke"][0]

smoke_confs    = []
no_smoke_confs = []

for img_path in val_smoke_imgs:
    res = best_model.predict(str(img_path), verbose=False, imgsz=224)
    smoke_confs.append(float(res[0].probs.data[smoke_idx]))

for img_path in val_no_smoke_imgs:
    res = best_model.predict(str(img_path), verbose=False, imgsz=224)
    no_smoke_confs.append(float(res[0].probs.data[smoke_idx]))

print(f"smoke    confidence → mean={np.mean(smoke_confs):.3f}  min={np.min(smoke_confs):.3f}")
print(f"no_smoke confidence → mean={np.mean(no_smoke_confs):.3f}  max={np.max(no_smoke_confs):.3f}")

# ── threshold별 지표 계산 ─────────────────────────────────
thresholds  = [i / 20 for i in range(1, 20)]   # 0.05 ~ 0.95
precisions, recalls, f1s = [], [], []

print(f"\n{'threshold':>10} | {'Precision':>10} | {'Recall':>8} | {'F1':>8} | TP | FP | FN | TN")
print("-" * 70)
best_f1, best_t = 0, 0
for t in thresholds:
    tp = sum(1 for c in smoke_confs    if c >= t)
    fn = sum(1 for c in smoke_confs    if c <  t)
    fp = sum(1 for c in no_smoke_confs if c >= t)
    tn = sum(1 for c in no_smoke_confs if c <  t)
    prec   = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1     = 2 * prec * recall / max(prec + recall, 1e-9)
    precisions.append(prec); recalls.append(recall); f1s.append(f1)
    marker = " ← best F1" if f1 > best_f1 else ""
    if f1 > best_f1: best_f1, best_t = f1, t
    print(f"  {t:8.2f}   | {prec:10.4f} | {recall:8.4f} | {f1:8.4f} | {tp:3d}| {fp:3d}| {fn:3d}| {tn:3d}{marker}")

print(f"\n최적 F1 threshold = {best_t:.2f}  (F1={best_f1:.4f})")

# ── PR Curve 시각화 ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: PR curve
axes[0].plot(recalls, precisions, "b-o", markersize=4)
for i, t in enumerate(thresholds):
    if t in (0.3, 0.5, 0.6, 0.7, best_t):
        axes[0].annotate(f"t={t:.2f}", (recalls[i], precisions[i]),
                         textcoords="offset points", xytext=(5, -10), fontsize=8)
axes[0].axvline(x=0.99, color="red", linestyle="--", alpha=0.5, label="Recall 목표 0.99")
axes[0].set_xlabel("Recall (smoke)"); axes[0].set_ylabel("Precision (smoke)")
axes[0].set_title("Precision-Recall Curve"); axes[0].legend(); axes[0].grid(True)

# 오른쪽: threshold별 Precision / Recall / F1
axes[1].plot(thresholds, precisions, "b-o", markersize=4, label="Precision")
axes[1].plot(thresholds, recalls,    "r-o", markersize=4, label="Recall")
axes[1].plot(thresholds, f1s,        "g-o", markersize=4, label="F1")
axes[1].axvline(x=best_t, color="gray", linestyle="--", label=f"Best F1 (t={best_t:.2f})")
axes[1].set_xlabel("Confidence Threshold"); axes[1].set_ylabel("Score")
axes[1].set_title("Threshold vs Metrics"); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig(str(BASE_DIR / "runs" / "smoke_detector" / "yolov8n_cls" / "pr_curve.png"), dpi=150)
plt.show()
print(f"PR curve 저장 완료")

smoke    confidence → mean=0.925  min=0.041
no_smoke confidence → mean=0.082  max=0.652

 threshold |  Precision |   Recall |       F1 | TP | FP | FN | TN
----------------------------------------------------------------------
      0.05   |     0.6895 |   0.9948 |   0.8145 | 191|  86|   1| 106 ← best F1
      0.10   |     0.7480 |   0.9896 |   0.8520 | 190|  64|   2| 128 ← best F1
      0.15   |     0.8008 |   0.9844 |   0.8832 | 189|  47|   3| 145 ← best F1
      0.20   |     0.8750 |   0.9844 |   0.9265 | 189|  27|   3| 165 ← best F1
      0.25   |     0.9300 |   0.9688 |   0.9490 | 186|  14|   6| 178 ← best F1
      0.30   |     0.9439 |   0.9635 |   0.9536 | 185|  11|   7| 181 ← best F1
      0.35   |     0.9686 |   0.9635 |   0.9661 | 185|   6|   7| 186 ← best F1
      0.40   |     0.9788 |   0.9635 |   0.9711 | 185|   4|   7| 188 ← best F1
      0.45   |     0.9892 |   0.9531 |   0.9708 | 183|   2|   9| 190
      0.50   |     0.9892 |   0.9531 |   0.9708 | 183|   2|   9| 190
    

C:\Users\User\AppData\Local\Temp\ipykernel_37448\754181925.py:68: UserWarning: Glyph 47785 (\N{HANGUL SYLLABLE MOG}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\User\AppData\Local\Temp\ipykernel_37448\754181925.py:68: UserWarning: Glyph 54364 (\N{HANGUL SYLLABLE PYO}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\User\AppData\Local\Temp\ipykernel_37448\754181925.py:69: UserWarning: Glyph 47785 (\N{HANGUL SYLLABLE MOG}) missing from font(s) DejaVu Sans.
  plt.savefig(str(BASE_DIR / "runs" / "smoke_detector" / "yolov8n_cls" / "pr_curve.png"), dpi=150)
C:\Users\User\AppData\Local\Temp\ipykernel_37448\754181925.py:69: UserWarning: Glyph 54364 (\N{HANGUL SYLLABLE PYO}) missing from font(s) DejaVu Sans.
  plt.savefig(str(BASE_DIR / "runs" / "smoke_detector" / "yolov8n_cls" / "pr_curve.png"), dpi=150)


<Figure size 1400x500 with 2 Axes>

PR curve 저장 완료


In [20]:
import matplotlib.pyplot as plt
import numpy as np

cm = np.array([[tp, fn],
               [fp, tn]])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
plt.colorbar(im, ax=ax)

classes = ["smoke (Positive)", "no_smoke (Negative)"]
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred: smoke", "Pred: no_smoke"], fontsize=11)
ax.set_yticks([0, 1]); ax.set_yticklabels(["GT: smoke", "GT: no_smoke"],     fontsize=11)

# 셀 값 표시
for i in range(2):
    for j in range(2):
        label = ["TP", "FN", "FP", "TN"][i * 2 + j]
        ax.text(j, i, f"{label}\n{cm[i,j]}", ha="center", va="center",
                fontsize=13, color="white" if cm[i,j] > cm.max() / 2 else "black")

ax.set_title(f"Confusion Matrix\nRecall(smoke)={recall:.4f}  F1={f1:.4f}", fontsize=12)
plt.tight_layout()
plt.savefig(str(BASE_DIR / "runs" / "smoke_detector" / "yolov8n_cls" / "confusion_matrix_custom.png"), dpi=150)
plt.show()
print("저장 완료")

<Figure size 500x400 with 2 Axes>

저장 완료


## 4. 목표 달성 판단 및 재학습 가이드

- **Recall ≥ 0.99** 달성 시 → 완료
- 미달 시 → conf threshold를 낮추거나 재학습 진행

In [21]:
TARGET_RECALL = 0.99

print("=" * 55)
if recall >= TARGET_RECALL:
    print(f"  [달성] Recall {recall:.4f} ≥ {TARGET_RECALL} — 목표 충족!")
    print(f"  최종 모델: {BEST_PT}")
else:
    gap = TARGET_RECALL - recall
    print(f"  [미달] Recall {recall:.4f} < {TARGET_RECALL} (부족분 {gap:.4f})")
    print()
    print("  ── 개선 옵션 ──────────────────────────────────")
    print(f"  옵션 A: conf threshold 낮추기 (현재 {CONF_THRES} → 0.1 시도)")
    print("  옵션 B: epochs 늘리기 (50 → 100)")
    print("  옵션 C: 클래스 가중치 추가 (pos_weight 설정)")
    print("  ──────────────────────────────────────────────")
print("=" * 55)

  [미달] Recall 0.7812 < 0.99 (부족분 0.2087)

  ── 개선 옵션 ──────────────────────────────────
  옵션 A: conf threshold 낮추기 (현재 0.3 → 0.1 시도)
  옵션 B: epochs 늘리기 (50 → 100)
  옵션 C: 클래스 가중치 추가 (pos_weight 설정)
  ──────────────────────────────────────────────
